In [1]:
import os
import sys

# Assicuriamoci che Python trovi la root directory (CLARITY)
# Se stiamo eseguendo questo notebook dalla cartella 'notebooks/', 
# dobbiamo salire di un livello per importare config.py e src/
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import config
from src.dataset import ClarityEvasionDataset

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split
import pickle

print(f"[*] MODALITÀ DEBUG ATTIVA? : {config.DEBUG_MODE}")
print(f"[*] MODELLO SELEZIONATO    : {config.MODEL_NAME}")
print(f"[*] MAX_LEN IMPOSTATA      : {config.MAX_LEN}")

# Device check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] DEVICE DISPONIBILE     : {device.upper()}")

e:\Riky\Documents\Poli\CLARITY\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] MODALITÀ DEBUG ATTIVA? : True
[*] MODELLO SELEZIONATO    : prajjwal1/bert-tiny
[*] MAX_LEN IMPOSTATA      : 128
[*] DEVICE DISPONIBILE     : CUDA


In [2]:
print(f"\n[*] Scaricamento del dataset da: {config.DATASET_URL}")
dataset = load_dataset(config.DATASET_URL)

print("\nStruttura del dataset originale:")
print(dataset)

# Convertiamo il train set in DataFrame Pandas
df_train_full = dataset['train'].to_pandas()

# In DEBUG_MODE prendiamo semplicemente le prime 'N' righe per testare il codice velocemente.
if config.DEBUG_MODE and config.SAMPLE_SIZE:
    print(f"\n[!] DEBUG MODE: Riduco il dataset a {config.SAMPLE_SIZE} righe casuali.")
    df_train_full = df_train_full.sample(n=config.SAMPLE_SIZE, random_state=42)

print(f"Dimensioni del DataFrame di lavoro: {df_train_full.shape}")


[*] Scaricamento del dataset da: ailsntua/QEvasion

Struttura del dataset originale:
DatasetDict({
    train: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 3448
    })
    test: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 308
    })
})

[!] DEBUG MODE: Riduco il dataset a 50 righe casuali.
Dimensioni del DataFrame di lavoro: (50, 20)


In [3]:
print("\n[*] Creazione dello split Train/Validation (80/20)...")

# Decidiamo dinamicamente se stratificare o no
# Se siamo in debug, stratify=None per evitare crash su classi con 1 solo elemento.
# Se siamo in run reale (Colab), stratifichiamo sulle 9 classi.
stratify_strategy = None if config.DEBUG_MODE else df_train_full['evasion_label']

train_df, val_df = train_test_split(
    df_train_full,
    test_size=0.20,
    random_state=42,
    stratify=stratify_strategy
)

# Convertiamo i Pandas DataFrame di nuovo nel formato Dataset di HuggingFace
from datasets import Dataset as HFDataset

hf_train = HFDataset.from_pandas(train_df).remove_columns(["__index_level_0__"])
hf_val = HFDataset.from_pandas(val_df).remove_columns(["__index_level_0__"])

print(f"Esempi nel Train set : {len(hf_train)}")
print(f"Esempi nel Val set   : {len(hf_val)}")

if not config.DEBUG_MODE:
    print("\nVerifica distribuzione 'evasion_label' nel Val set (dovrebbe specchiare il Train):")
    print(val_df['evasion_label'].value_counts(normalize=True).round(3) * 100)
else:
    print("\n[!] Distribuzione non verificata (Stratificazione disattivata in DEBUG_MODE).")


[*] Creazione dello split Train/Validation (80/20)...
Esempi nel Train set : 40
Esempi nel Val set   : 10

[!] Distribuzione non verificata (Stratificazione disattivata in DEBUG_MODE).


In [4]:
print(f"\n[*] Caricamento del tokenizer per: {config.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

print("\n[*] Creazione dei Dataset PyTorch Custom...")
# Passiamo la mappatura dal config (EVASION2ID) per convertire le stringhe in numeri (0-8)
train_dataset = ClarityEvasionDataset(
    hf_dataset=hf_train,
    tokenizer=tokenizer,
    max_len=config.MAX_LEN,
    evasion2id=config.EVASION2ID
)

val_dataset = ClarityEvasionDataset(
    hf_dataset=hf_val,
    tokenizer=tokenizer,
    max_len=config.MAX_LEN,
    evasion2id=config.EVASION2ID
)

# Estraiamo un campione per vedere come la nostra classe ha preparato i tensori
sample = train_dataset[0]
print("\nEsempio di output dal custom Dataset (primo elemento del Train set):")
print(f"Input IDs shape      : {sample['input_ids'].shape}")
print(f"Attention Mask shape : {sample['attention_mask'].shape}")
print(f"Label ID (Task 2)    : {sample['labels']} -> Corrisponde a: {config.ID2EVASION[sample['labels'].item()]}")

# Decodifichiamo i primi 50 token per vedere come il modello "legge" il nostro prompt
testo_decodificato = tokenizer.decode(sample['input_ids'][:50])
print(f"\nCome il modello legge il testo (primi 50 token):\n{testo_decodificato}")


[*] Caricamento del tokenizer per: prajjwal1/bert-tiny


e:\Riky\Documents\Poli\CLARITY\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



[*] Creazione dei Dataset PyTorch Custom...

Esempio di output dal custom Dataset (primo elemento del Train set):
Input IDs shape      : torch.Size([128])
Attention Mask shape : torch.Size([128])
Label ID (Task 2)    : 1 -> Corrisponde a: Implicit

Come il modello legge il testo (primi 50 token):
[CLS] question : responsibility of president putin for the intransigence of governments in ukraine and syria [SEP] context : q. the common denominator in the strife in ukraine and syria is the support that those two governments get from russia, and i '


In [5]:
# Usiamo pickle per salvare gli oggetti dataset interi
train_path = os.path.join(config.DATA_DIR, "train_dataset.pkl")
val_path = os.path.join(config.DATA_DIR, "val_dataset.pkl")

print(f"\n[*] Salvataggio dei Dataset processati nella cartella '{config.DATA_DIR}'...")

with open(train_path, 'wb') as f:
    pickle.dump(train_dataset, f)

with open(val_path, 'wb') as f:
    pickle.dump(val_dataset, f)

print("[✓] Salvataggio completato con successo. Puoi passare al Notebook 02_Model_Training.ipynb.")


[*] Salvataggio dei Dataset processati nella cartella 'e:\Riky\Documents\Poli\CLARITY\notebooks\data'...
[✓] Salvataggio completato con successo. Puoi passare al Notebook 02_Model_Training.ipynb.
